# Native PyTorch tutorial 1

5,000 AdaBelief epochs with gradient centralization, followed by 200 L-BFGS rounds of up to 1,000 iterations. Native build/model return the same trainable module. Logs and checkpoints are written by the visible training loop; GPU selection uses PyTorch devices.

Run all cells in order. Full scientific workloads are the default; `NIF_SMOKE=1` is a separate development run and does not establish full convergence. The original TensorFlow and `_colab` notebooks are preserved as legacy references.

In [ ]:
# Run this cell first in a clean Colab Python 3.11–3.13 session.
import os, sys, subprocess, importlib.util
from pathlib import Path
# A local checkout is used directly for development; Colab installs the public pytorch branch.
repo = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "nif" / "model.py").exists()), None)
if repo is not None:
    sys.path.insert(0, str(repo))
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
        "torch==2.14.1", "numpy==2.1.3", "scipy==1.16.3", "matplotlib==3.10.0",
        "protobuf==7.36.2", "google-crc32c==1.9.0", "tfrecord==1.14.6",
        "git+https://github.com/pswpswpsw/nif.git@pytorch"])
import copy, json, time, math
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from matplotlib import pyplot as plt
import nif
# FULL is the default. Set NIF_SMOKE=1 before starting only for a short development check.
SMOKE = os.environ.get("NIF_SMOKE", "0") == "1"
device = torch.device(os.environ.get("NIF_DEVICE", "cuda" if torch.cuda.is_available() else "cpu"))
torch.manual_seed(42)
np.random.seed(42)
if device.type == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
print({"torch": torch.__version__, "device": str(device), "mode": "SMOKE" if SMOKE else "FULL"})

OUT = Path(os.environ.get("NIF_OUTPUT_DIR", "pytorch_outputs")) / "tutorial1"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Explicit native loops replace compile/fit/callbacks. The loss reduction is per row,
# then the mean over the batch; area weights multiply each row's MSE (not weight-normalized).
def dense_tanh(inputs, outputs):
    # Preserve the original Dense default: Glorot uniform kernel and zero bias.
    layer = nn.Linear(inputs, outputs)
    nn.init.xavier_uniform_(layer.weight)
    nn.init.zeros_(layer.bias)
    return nn.Sequential(layer, nn.Tanh())

def epochs(full):
    return 2 if SMOKE else full

def tensor(array, on_device=device):
    return torch.as_tensor(array, dtype=torch.float32, device=on_device)

def loader(features, target, batch_size, weight=None):
    arrays = [tensor(features, "cpu"), tensor(target, "cpu")]
    if weight is not None:
        arrays.append(tensor(weight, "cpu").reshape(-1))
    return DataLoader(TensorDataset(*arrays), batch_size=batch_size, shuffle=True,
                      drop_last=False, pin_memory=device.type == "cuda")

def predict(model, features, batch_size=1024):
    model.eval()
    result = []
    with torch.no_grad():
        for xb in torch.as_tensor(features, dtype=torch.float32).split(batch_size):
            result.append(model(xb.to(next(model.parameters()).device)).detach().cpu())
    return torch.cat(result).numpy()

def centralize_gradients(model):
    # TF kernels are [input,output]; native Linear stores [output,input].
    # Centralize along the input axes, preserving the original mathematical operation.
    for module in model.modules():
        for name, parameter in module.named_parameters(recurse=False):
            if parameter.grad is not None and parameter.ndim > 1:
                axes = tuple(range(1, parameter.ndim)) if isinstance(module, nn.Linear) else tuple(range(parameter.ndim-1))
                parameter.grad.sub_(parameter.grad.mean(dim=axes, keepdim=True))

def save_checkpoint(model, optimizer, path, epoch):
    payload = {"model_state": {k: v.detach().cpu() for k,v in model.state_dict().items()},
               "optimizer_state": optimizer.state_dict(), "epoch": epoch,
               "framework": "pytorch", "smoke": SMOKE}
    if hasattr(model, "cfg_shape_net"):
        payload.update(model_class=type(model).__name__, cfg_shape_net=model.cfg_shape_net,
                       cfg_parameter_net=model.cfg_parameter_net, mixed_policy="float32")
    torch.save(payload, path)

def save_and_check(model, optimizer, name, example, factory=None):
    path = OUT / (name + ".pt")
    save_checkpoint(model, optimizer, path, -1)
    checkpoint = torch.load(path, map_location=device, weights_only=True)
    if factory is None:
        factory = lambda: type(model)(copy.deepcopy(model.cfg_shape_net),
                                       copy.deepcopy(model.cfg_parameter_net), "float32")
    restored = factory().to(device)
    restored.load_state_dict(checkpoint["model_state"])
    restored.eval(); model.eval()
    with torch.no_grad():
        expected, actual = model(tensor(example)), restored(tensor(example))
    torch.testing.assert_close(actual, expected, rtol=1e-5, atol=1e-6)
    print("Checkpoint predictions verified:", path)
    return restored

def train(model, batches, optimizer, nepoch, name, schedule=None,
          centralization=False, loss_function=None, before_epoch=None):
    history = []
    start = time.monotonic()
    for epoch in range(nepoch):
        model.train()
        if schedule is not None:
            for group in optimizer.param_groups:
                group["lr"] = schedule(epoch, group["lr"])
        if before_epoch is not None:
            before_epoch(epoch)
        summed, seen = {}, 0
        for batch in batches:
            xb, yb = (item.to(device) for item in batch[:2])
            optimizer.zero_grad(set_to_none=True)
            if loss_function is None:
                per_row = (model(xb) - yb).square().mean(dim=-1)
                if len(batch) == 3:
                    per_row = per_row * batch[2].to(device).reshape(-1)
                data_loss = per_row.mean()
                values = {"mse": data_loss}
            else:
                data_loss, values = loss_function(model, xb, yb)
            regularization = model.regularization_loss(xb) if hasattr(model, "regularization_loss") else xb.new_zeros(())
            loss = data_loss + regularization
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss in {name}, epoch {epoch}")
            loss.backward()
            if centralization:
                centralize_gradients(model)
            optimizer.step()
            for key, value in {**values, "regularization": regularization, "loss": loss}.items():
                summed[key] = summed.get(key, 0.0) + float(value.detach()) * len(xb)
            seen += len(xb)
        row = {"epoch": epoch, "rows": seen, **{key: value / seen for key,value in summed.items()}}
        history.append(row)
        if epoch % 100 == 0 or epoch == nepoch - 1:
            print(name, row, "elapsed_seconds", round(time.monotonic()-start, 2))
            (OUT / (name + "_history.json")).write_text(json.dumps(history, indent=2))
        if epoch % 1000 == 0 or epoch == nepoch - 1:
            save_checkpoint(model, optimizer, OUT / (name + "_latest.pt"), epoch)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.semilogy([row["loss"] for row in history]); ax.set(xlabel="epoch", ylabel="loss", title=name)
    fig.tight_layout(); fig.savefig(OUT / (name + "_loss.png")); plt.show(); plt.close(fig)
    return history

def wave_figure(truth, prediction, name):
    fig, axs = plt.subplots(1,3,figsize=(15,3))
    for ax, values, title in zip(axs, (truth,prediction,prediction-truth), ("true","pred","error")):
        im=ax.contourf(tt,xx,values.reshape(10,200),levels=50,cmap="seismic")
        fig.colorbar(im,ax=ax); ax.set(xlabel="t",ylabel="x",title=title)
    fig.tight_layout(); fig.savefig(OUT / (name+"_reconstruction.png")); plt.show(); plt.close(fig)

def write_metrics(values):
    print(json.dumps(values, indent=2))
    (OUT / "metrics.json").write_text(json.dumps({"mode": "smoke" if SMOKE else "full", **values}, indent=2))


# Generate 1D travelling wave data

In [ ]:
NT=10 # 20
NX=200

x = np.linspace(0,1,NX,endpoint=False)
t = np.linspace(0,100,NT,endpoint=False)

xx,tt=np.meshgrid(x,t)

omega = 4
c = 0.12/20
x0 = 0.2

u = np.exp(-1000*(xx-x0-c*tt)**2)*np.sin(omega*(xx-x0-c*tt))

# vis
plt.figure()
for i in range(NT):
    plt.plot(x,u[i,:],'-',label=str(i) + '-th time')

plt.xlabel('$x$',fontsize=25)
plt.ylabel('$u$',fontsize=25)

# vis iso
plt.figure(figsize=(4,4))
ax = plt.axes(projection='3d')
ax.plot_surface(xx,tt,u,cmap="rainbow", lw=2)#,rstride=1, cstride=1)
ax.view_init(57, -80)
ax.set_xlabel(r'$x$',fontsize=25)
ax.set_ylabel(r'$t$',fontsize=25)
ax.set_zlabel(r'$u$',fontsize=25)

plt.tight_layout()
# Keep all three axis labels inside the ordinary inline tight crop.
ax.set_box_aspect(None, zoom=0.75)
ax.xaxis.labelpad = ax.yaxis.labelpad = ax.zaxis.labelpad = 12


# Fitting $u(x,t) \approx u_{nif}(x,t) = u_{shape}(x; \Theta_{parameter}(t))$ 
where $u_{shape}$ and $\Theta_{parameter}$ are two feedforward neural networks



* We define a shape-net with 1 input $x$, 1 output $u$, 2 hidden layers, Swish activation function, each layer has 30 units

  * note that here we assume parameter-net fully parameterizes the shape-net, so `connectivity` is `full`

* Then we define a parameter-net, still 1 input, $t$, and 1 latent dimension, 2 layers with each layer 30 units, also activated by Swish



In [ ]:
cfg_shape_net = {"connectivity": "full", "input_dim": 1, "output_dim": 1,
                 "units": 30, "nlayers": 2, "activation": "swish"}
cfg_parameter_net = {"input_dim": 1, "latent_dim": 1, "units": 30,
                     "nlayers": 2, "activation": "swish"}


* we can setup whether to enable
  * **multiple gpus**,
  * **mixed precision (fp32 for store and fp16 for computing)**

* However at Colab, we don't have multiple-gpu (unless we pay). So it won't be effective here. 

* But you can play around with mixed precision, if your GPU supports it. Typically I found there is a slight increase in training speed. 

In [ ]:
enable_multi_gpu = False
enable_mixed_precision = False
mixed_policy = "float32"
# Set device=NIF_DEVICE/cuda for GPU execution; this full workload uses FP32.


* now we can further setup 
  * the total number of epochs
  * initial learning rate
  * batch size
  * number of epochs for check point saving
  * number of epochs for logging error file
  * number of epochs for plotting figures

In [ ]:
nepoch = 5000
lr = 5e-3
batch_size = 512
checkpt_epoch, display_epoch, print_figure_epoch = 1000, 100, 100


* now let's read the data from the disk
* then we create a a `DataLoader` with `features` as `t,x` and `target` as `u`. everything is point-wise.
* we will further shuffle, batching, and pre-fetch it so as to optimize the speed, in which case, we won't have data I/O as out bottleneck 

In [ ]:
# alternatively, get the training dataset from nif.demo for this example
from nif.demo import TravelingWave
tw = TravelingWave()
train_data = tw.data
train_dataset = loader(train_data[:, :2], train_data[:, -1:], batch_size)


* Here we define another training-loop hook to modulate the learning rate, it is intuitive and heuristic, so you need to learn how to tune it a bit. but most of the time it is okay. indeed, in our nif paper, we didn't do any tuning but here I am doing this just to show that we can. 

In [ ]:
def scheduler(epoch, lr):
    if epoch < 1000: return lr
    if epoch < 2000: return 1e-3
    if epoch < 4000: return 5e-4
    return 1e-4


In [ ]:
from nif.optimizers import AdaBeliefOptimizer
model_ori = nif.NIF(cfg_shape_net, cfg_parameter_net, mixed_policy).to(device)
model_opt = model_ori.build()
assert model_opt is model_ori
optimizer = AdaBeliefOptimizer(model_opt.parameters(), lr=lr)
history = train(model_opt, train_dataset, optimizer, epochs(nepoch), "traveling_wave",
                schedule=scheduler, centralization=True)
wave_figure(train_data[:, -1], predict(model_opt, train_data[:, :2]).ravel(), "traveling_wave")


* Finally, you can just visualize the loss and predictions

# Get the mapping from parameter to latent representation
* Note it is only part of the ParameterNet

In [ ]:
model_p_to_lr = model_ori.model_p_to_lr()
print(model_p_to_lr)
p = tensor(train_data[:32, :1])
latent = model_p_to_lr(p)
# Plot every training time separately from the 32-row shared-subnet example.
plot_p = tensor(np.unique(train_data[:, 0]).reshape(-1, 1))
with torch.no_grad():
    plot_latent = model_p_to_lr(plot_p)
fig, ax=plt.subplots(); ax.plot(plot_p.cpu(),plot_latent.cpu(),"o"); ax.set(xlabel="normalized t",ylabel="latent")
fig.savefig(OUT / "latent.png"); plt.show(); plt.close(fig)


* You can also save such model separately
 - note that here you don't have to save the checkpoints for those **derived model**, because anyway they won't be used to further train at all, unless for special purpose

In [ ]:
torch.save(model_p_to_lr.state_dict(), OUT / "parameter_to_latent.pt")


# Get the mapping from latent representation to weights and biases

In [ ]:
model_lr_to_w = model_ori.model_lr_to_w()
w = model_lr_to_w(latent)
torch.testing.assert_close(w, model_ori.model_p_to_w()(p))
print("latent-to-weights:", w.shape)


# Get ShapeNet conditioned on weights and biases

* there SHOULD NOT be any trainable variables. the below is just to give a sanity-check

In [ ]:
model_x_to_u_given_w = model_ori.model_x_to_u_given_w()
torch.testing.assert_close(model_x_to_u_given_w([tensor(train_data[:32,1:2]), w]),
                           model_opt(tensor(train_data[:32,:2])))
print(model_x_to_u_given_w)


# Restore model from checkpoint

1. construct a NIF instance (what I previously call *hub of model*) with the **same** configurations for shape net and parameter net, as the one consistent with the checkpoint


2. Reconstruct the native module from its saved configuration and load its state dictionary. Verify predictions after restoration; no compilation is needed.

3. Now, we load weights from the desired checkpoint directory

# Fine-tuning with L-BFGS
* **it is not used at all in our paper, but just to show that if you want you can**
* early researchers have tried using L-BFGS instead of standard GD/SGD. it does NOT work well in the early stage training deep nets. but in the late stage of training, many works on PINN have reported its usefulness.

In [ ]:
new_model = save_and_check(model_opt, optimizer, "traveling_wave", train_data[:64,:2])


- first we prepare the feature and labels

- then we use native `torch.optim.LBFGS` to wrap around our model, get an instance that can further fine tune the model

In [ ]:
data_feature = tensor(train_data[:, :2])
data_label = tensor(train_data[:, -1:])


- now we can fine tune the model with `.minimize(rounds, max_iter)`
  - `max_iter` means the we will run LBFGS updates until maximal number of iterations is achieved or error tolerance is achieved
  - here `rounds` means repeating the above process. it can be useful to force running optimization process when error doesn't change even at the level of 1e-15.  

In [ ]:
# Native torch.optim.LBFGS uses a closure: recompute loss and gradients at every line search.
lbfgs_history = []
rounds, max_iter = (1, 2) if SMOKE else (200, 1000)
# Public TFPLBFGS scientific controls, retained in full AND smoke modes.
tolerance = x_tolerance = f_relative_tolerance = 1e-15
num_correction_pairs = 20
max_line_search_iterations = 100
# Native convention (the same mapping as nif.optimizers.TFPLBFGS):
# PyTorch uses strong-Wolfe line search and a per-round max_eval budget of
# max_iter * max_line_search_iterations. Its internal trial accounting is not
# TFP's separate limit of 100 iterations per individual line search.
# tolerance_change applies to absolute step/objective changes (and the descent
# directional derivative); it cannot express TFP's independent relative-objective
# tolerance. Both original change thresholds are 1e-15 and map to that value.
# Thus the controls are retained, but solver trajectories/stopping can differ.


## Finally, let's visualize the fine-tuned results

In [ ]:
for round_index in range(rounds):
    fine_tuner = torch.optim.LBFGS(new_model.parameters(), max_iter=max_iter,
                                 max_eval=max_iter * max_line_search_iterations,
                                 history_size=num_correction_pairs,
                                 tolerance_grad=tolerance,
                                 tolerance_change=max(x_tolerance, f_relative_tolerance),
                                 line_search_fn="strong_wolfe")
    def closure():
        fine_tuner.zero_grad(set_to_none=True)
        loss = (new_model(data_feature)-data_label).square().mean() + new_model.regularization_loss(data_feature)
        if not torch.isfinite(loss): raise FloatingPointError("non-finite LBFGS loss")
        loss.backward(); lbfgs_history.append(float(loss.detach()))
        return loss
    fine_tuner.step(closure)
new_model = save_and_check(new_model, fine_tuner, "fine_tuned", train_data[:64,:2])
fig,ax=plt.subplots(figsize=(8,2)); ax.semilogy(lbfgs_history); ax.set(xlabel="closure evaluation",ylabel="loss")
fig.tight_layout(); fig.savefig(OUT / "fine_tune_loss.png"); plt.show(); plt.close(fig)


In [ ]:
prediction = predict(new_model, train_data[:, :2]).ravel()
wave_figure(train_data[:,-1], prediction, "fine_tuned")
write_metrics({"adabelief_epochs": epochs(nepoch), "lbfgs_rounds": rounds, "lbfgs_max_iter": max_iter,
               "final_mse": float(np.mean((prediction-train_data[:,-1])**2))})
